# Ablation of latent-space features

Analyse des features en sortie de codebook du backbone gelé, avec plusieurs points de vue :
1. **Géométrie cosinus** (intra/inter classe, centroïdes)
2. **Séparabilité** (nearest-centroid, kNN, sonde linéaire)
3. **Riemannien** (covariances SPD, MDM, tangent space)
4. **Anisotropie / rang effectif** (collapse de la représentation)

In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))
import numpy as np
import torch
import matplotlib.pyplot as plt
import yaml
from pathlib import Path
from model import EEGVAE
from eeg_preprocessing.loaders import CustomDataset, get_subject_ids
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BACKBONE_CKPT = "/home/infres/yrothlin-24/EEG_Foundation_Model/outputs/pretraining/without_augmentation/pretraining_best.pt"

DATASETS = {
    "BCI2A":       ("/projects/EEG-foundation-model/RECH202/tests_2/BCI2A/bci2a.lmdb", 4),
    "SEEDV":       ("/projects/EEG-foundation-model/RECH202/tests_2/SEEDV/seedv.lmdb", 5),
    "PHYSIONETMI": ("/projects/EEG-foundation-model/RECH202/tests_2/PHYSIONETMI/physionetmi.lmdb", 4),
}
N_SUBJECTS = 4

In [ ]:
def load_backbone(ckpt_path):
    cfg = yaml.safe_load(open("../configs/model.yaml"))["model"]
    sd = torch.load(ckpt_path, map_location="cpu")["model"]
    model = EEGVAE(**cfg).to(device).eval()
    model.load_state_dict(sd)
    for p in model.parameters():
        p.requires_grad = False
    return model

model = load_backbone(BACKBONE_CKPT)

In [ ]:
from pyriemann.estimation import Covariances

@torch.no_grad()
def extract(lmdb_path, n_subjects=N_SUBJECTS, channel_mode="mapped"):
    cov_est = Covariances(estimator="oas")
    pooled, covs, labels, subjects = [], [], [], []
    for sid in get_subject_ids(Path(lmdb_path))[:n_subjects]:
        ds = CustomDataset(lmdb_path, subject_ids=[sid], channel_mode=channel_mode)
        for x, y in DataLoader(ds, batch_size=128, num_workers=4):
            z = model.encode(x.to(device).float()).mode()      # (B, C, T)
            pooled.append(z.mean(-1).cpu().numpy())
            covs.append(cov_est.transform(z.cpu().numpy()))
            labels.append(y.numpy())
            subjects.append(np.full(len(y), sid))
    return (np.concatenate(pooled), np.concatenate(covs),
            np.concatenate(labels), np.concatenate(subjects))

DATA = {}
for name, (lmdb, ncls) in DATASETS.items():
    p, c, y, s = extract(lmdb)
    DATA[name] = dict(pooled=p, cov=c, y=y, subj=s, n_classes=ncls)
    print(f"{name}: pooled={p.shape} cov={c.shape} classes={len(np.unique(y))} n={len(y)}")

## 1. Géométrie cosinus

In [ ]:
def cosine_split(feat, y):
    E = feat / (np.linalg.norm(feat, axis=1, keepdims=True) + 1e-9)
    S = E @ E.T
    iu = np.triu_indices(len(E), 1)
    same = y[iu[0]] == y[iu[1]]
    return S[iu][same], S[iu][~same]

n = len(DATA)
fig, axes = plt.subplots(1, n, figsize=(6 * n, 5), squeeze=False)
for ax, (name, d) in zip(axes[0], DATA.items()):
    intra, inter = cosine_split(d["pooled"], d["y"])
    ax.hist(inter, bins=80, density=True, alpha=0.6, color="#1f77b4", label="inter")
    ax.hist(intra, bins=80, density=True, alpha=0.6, color="#d62728", label="intra")
    ax.axvline(intra.mean(), color="#d62728", ls="--", lw=1)
    ax.axvline(inter.mean(), color="#1f77b4", ls="--", lw=1)
    ax.set_title(f"{name}  gap={intra.mean()-inter.mean():+.4f}")
    ax.set_xlabel("cosine similarity"); ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, n, figsize=(5 * n, 4), squeeze=False)
for ax, (name, d) in zip(axes[0], DATA.items()):
    E = d["pooled"] / (np.linalg.norm(d["pooled"], axis=1, keepdims=True) + 1e-9)
    cls = np.unique(d["y"])
    C = np.stack([E[d["y"] == k].mean(0) for k in cls])
    C = C / (np.linalg.norm(C, axis=1, keepdims=True) + 1e-9)
    im = ax.imshow(C @ C.T, vmin=-1, vmax=1, cmap="coolwarm")
    ax.set_title(f"{name} centroid cosine")
    ax.set_xticks(range(len(cls))); ax.set_yticks(range(len(cls)))
    fig.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout(); plt.show()

## 2. Séparabilité : nearest-centroid, kNN, sonde linéaire

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.neighbors import NearestCentroid

def probe_scores(feat, y):
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
    out = {}
    out["nearest-centroid"] = cross_val_score(
        make_pipeline(StandardScaler(), NearestCentroid()), feat, y, cv=cv).mean()
    out["kNN"] = cross_val_score(
        make_pipeline(StandardScaler(), KNeighborsClassifier(15)), feat, y, cv=cv).mean()
    out["linear"] = cross_val_score(
        make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)), feat, y, cv=cv).mean()
    return out

rows = {}
for name, d in DATA.items():
    rows[name] = probe_scores(d["pooled"], d["y"])
    rows[name]["chance"] = 1.0 / d["n_classes"]
    print(name, {k: round(v, 3) for k, v in rows[name].items()})

## 3. Riemannien : covariances SPD + MDM (comparé à l'euclidien)

In [ ]:
from pyriemann.classification import MDM

def riemann_score(cov, y):
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
    return cross_val_score(MDM(metric="riemann"), cov, y, cv=cv).mean()

for name, d in DATA.items():
    rows[name]["riemann-MDM"] = riemann_score(d["cov"], d["y"])
    print(name, "riemann-MDM =", round(rows[name]["riemann-MDM"], 3),
          "| chance =", round(rows[name]["chance"], 3))

In [ ]:
methods = ["nearest-centroid", "kNN", "linear", "riemann-MDM"]
fig, ax = plt.subplots(figsize=(2.2 * len(DATA) + 3, 5))
x = np.arange(len(DATA)); w = 0.2
for i, m in enumerate(methods):
    ax.bar(x + i * w, [rows[n][m] for n in DATA], w, label=m)
for j, name in enumerate(DATA):
    ax.hlines(rows[name]["chance"], j - 0.1, j + 3 * w + 0.1, colors="k", ls=":")
ax.set_xticks(x + 1.5 * w); ax.set_xticklabels(list(DATA))
ax.set_ylabel("5-fold accuracy"); ax.set_title("class separability (dotted = chance)")
ax.legend(); plt.tight_layout(); plt.show()

## 4. Tangent space riemannien (visualisation 2D)

In [ ]:
from pyriemann.tangentspace import TangentSpace
from sklearn.decomposition import PCA

fig, axes = plt.subplots(1, len(DATA), figsize=(6 * len(DATA), 5), squeeze=False)
for ax, (name, d) in zip(axes[0], DATA.items()):
    ts = TangentSpace(metric="riemann").fit_transform(d["cov"])
    xy = PCA(n_components=2).fit_transform(ts)
    for k in np.unique(d["y"]):
        m = d["y"] == k
        ax.scatter(xy[m, 0], xy[m, 1], s=6, alpha=0.5, label=str(k))
    ax.set_title(f"{name} tangent-space PCA"); ax.legend(markerscale=2, fontsize=8)
plt.tight_layout(); plt.show()

## 5. Anisotropie / rang effectif (collapse de la représentation)

In [ ]:
def participation_ratio(feat):
    X = feat - feat.mean(0)
    ev = np.linalg.svd(X, compute_uv=False) ** 2
    return (ev.sum() ** 2) / (np.sum(ev ** 2) + 1e-12)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for name, d in DATA.items():
    X = d["pooled"] - d["pooled"].mean(0)
    ev = np.linalg.svd(X, compute_uv=False) ** 2
    ev = ev / ev.sum()
    axes[0].plot(np.cumsum(ev), label=name)
axes[0].set_xlabel("component"); axes[0].set_ylabel("cumulative explained var")
axes[0].set_title("PCA spectrum"); axes[0].legend()

pr = {name: participation_ratio(d["pooled"]) for name, d in DATA.items()}
axes[1].bar(list(pr), list(pr.values()))
axes[1].set_ylabel("participation ratio (effective rank)")
axes[1].set_title(f"effective dim (of {DATA[list(DATA)[0]]['pooled'].shape[1]})")
plt.tight_layout(); plt.show()
print({k: round(v, 1) for k, v in pr.items()})

## Lecture

- **gap cosinus intra-inter ≈ 0** + **nearest-centroid > chance** → signal de classe présent mais **au niveau centroïde**, pas par-paire → mauvais terrain pour un contrastif par paires (SupCon/InfoNCE).
- **riemann-MDM vs euclidien** : si MDM > les probes euclidiennes, l'info discriminante est surtout dans la **structure de covariance**, pas dans la moyenne temporelle poolée.
- **participation ratio faible** (≪ nombre de canaux) → représentation **anisotrope / partiellement collapsée**.